# Would this method have found an effect in a world with none?

A fitted model will always produce an estimate. The question that separates a finding from an
artifact is what the same procedure does when the effect is not there — shuffle the treatment
across units, permute the assignment, drop a third of the panel, add noise. A real effect
survives all four. A pipeline artifact usually survives none of them, and nobody notices
because nobody runs them.

The second half is forecasting, and it hides a subtler trap: evaluating a dynamic model only on
the horizon, with the carryover state that the *test* period implies rather than the one the
training window ended in. That flatters a model with memory, which is exactly the kind this
package fits.

**Refutation** asks whether an estimate behaves the way a real effect should: vanish under a
placebo treatment, beat its permutation null, and stay put under subsampling and added noise.
Every check returns a `Refutation` with the original estimate, the refuted distribution's
interval, a p-value with its `N`, and `passed` under a stated rule.

**Backtests** ask whether the fitted surface forecasts. `rolling_origin` refits at each origin
and forecasts `horizon` periods through the same `forward` the likelihood used, with the
carryover state carried in from the training window — the "graph-faithful" fix for the parent
repo's horizon-only evaluation. `freeze` pins a fit so it can be replayed on new panels.

In [ ]:
import numpy as np

from axiom.core import Posterior, PredictiveDraws, Unsupported
from axiom.diagnose import (
    Backtest, FrozenPredictor, HorizonScore, OriginFailure, OriginForecast, Refutation, RefutationKind,
    added_noise, crps, forecast, freeze, permutation, placebo_treatment, random_subset, rolling_origin,
    training_panel,
)
from axiom.sim import DosePlan, surface_world
from axiom.surface import GeometricCarryover, fit

from axiom.display import enable, table
from axiom.viz import backtest_plot

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, CRITICAL, ORANGE, caption, compare, curve_band, intervals, mark_x

enable();  # every axiom result renders itself from here on

DRAWS = 40
world = surface_world(n_units=5, n_periods=8, treatments=("a",), intercept="shared", doses=DosePlan(scale=50.0, zero_fraction=0.3), noise_sd=0.1, seed=21)
res = fit(world.spec, world.panel, backend="laplace", draws=DRAWS, chains=1, seed=2)
print("converged:", res.converged)

## Placebo and permutation

`placebo_treatment` reassigns the treatment's dose path across units and refits: the effect
should vanish. `permutation` builds the null distribution of the estimand from `n` refits and
reports the permutation p-value `(n_extreme + 1) / (n + 1)`, so `n` must be large enough for
`alpha` to be reachable.

In [ ]:
placebo = placebo_treatment(res, seed=0, draws=DRAWS, panel=world.panel)
assert isinstance(placebo, Refutation)
kind: RefutationKind = placebo.kind
print(f"{kind}: original {placebo.original:.3f} {placebo.original_interval} -> placebo mean {placebo.refuted_mean:.3f}")
print("rule:", placebo.rule)
print("p =", round(placebo.p_value, 3), "| passed:", placebo.passed, "| round-trips:", Refutation.from_json(placebo.to_json()) == placebo)

perm = permutation(res, n=5, seed=0, alpha=0.2, draws=DRAWS, panel=world.panel)
assert isinstance(perm, Refutation)
print(f"permutation: null estimates {np.round(perm.refuted_estimates, 3)} p={perm.p_value:.3f} passed={perm.passed}")

## Stability: random subsets and added noise

`random_subset` refits on random subsets of units; `added_noise` refits after adding Gaussian
noise at `sd_fraction` of the fitted noise sd. Both pass when the original is not in the tails
of the perturbed estimates.

In [ ]:
subset = random_subset(res, fraction=0.6, n=3, seed=0, alpha=0.1, draws=DRAWS, panel=world.panel)
assert isinstance(subset, Refutation)
print(f"random_subset: {subset.rule}\n  estimates {np.round(subset.refuted_estimates, 3)} passed={subset.passed}")
noisy = added_noise(res, sd_fraction=0.5, n=3, seed=0, alpha=0.1, draws=DRAWS, panel=world.panel)
assert isinstance(noisy, Refutation)
print(f"added_noise: sigma_hat={float(noisy.detail['sigma_hat']):.3f} noise_sd={float(noisy.detail['noise_sd']):.3f} passed={noisy.passed}")

In [ ]:
checks = {"placebo treatment": placebo, "permutation null": perm, "random subset": subset, "added noise": noisy}
rows = []
for label, ref in checks.items():
    values = np.asarray(ref.refuted_estimates, dtype=float)
    rows.append((f"{label}  ({'passed' if ref.passed else 'FAILED'})",
                 float(values.mean()), float(values.min()), float(values.max())))
fig = intervals(
    rows, ref=float(placebo.original), ref_label="the estimate being defended",
    title="Four ways to try to break the same number",
    subtitle="range of the refuted estimates under each check, against the original",
    x_title="estimated effect",
)
caption(fig, "A placebo or permutation run that lands on the original estimate means the "
             "procedure would have produced that number from noise. These land well below it, "
             "and the subset and noise refits land near it — which is the pattern a real "
             "effect makes and an artifact does not.")

## Forecasting through the one `forward`

`training_panel` is the prefix of a panel; `forecast` evaluates the surface over the *full*
dose path at posterior draws and returns the horizon slice, so the carryover state entering
the horizon is the real one. At the world's true parameters the forecast equals the world's
own `forward`. `crps` scores a draw ensemble against observations (and degenerates to the
absolute error for a point forecast).

In [ ]:
carry = surface_world(n_units=3, n_periods=12, treatments=("a",), carryover={"a": GeometricCarryover(max_lag=4)}, intercept="shared", doses=DosePlan(scale=50.0, zero_fraction=0.2), noise_sd=0.1, seed=31)
train = training_panel(carry.panel, 8)
print("training periods:", [int(p) for p in train.periods])
truth = Posterior({k: np.asarray(v)[None, None, ...] for k, v in carry.theta.items()})
fc = forecast(carry.spec, truth, carry.panel, origin=8, horizon=3)
assert isinstance(fc, PredictiveDraws)
print("forecast shape (chain, draw, unit, step):", fc.values.shape)
print("equals the world's forward on the horizon:", np.allclose(fc.values[0, 0], carry.forward()[:, 8:11]))
ensemble = np.random.default_rng(0).normal(size=(50, 4))
print("crps of an ensemble:", np.round(crps(ensemble, np.zeros(4)), 3), "| point forecast = |error|:", crps(np.full((1, 2), 2.0), np.full(2, 0.5)))

## `rolling_origin`

Each origin gets a refit on its prefix and an `OriginForecast` (observed, mean, lower, upper
per unit and step); each horizon step gets a `HorizonScore` with MAE, RMSE, CRPS, bias,
interval coverage, and the exact acceptance region for that coverage's `N`. Failed refits are
`OriginFailure` records, never silent drops.

In [ ]:
bt = rolling_origin(carry.spec, carry.panel, origins=(8, 10), horizon=2, draws=DRAWS, chains=1, seed=0)
assert isinstance(bt, Backtest)
rows = []
for sc in bt.scores:
    assert isinstance(sc, HorizonScore)
    rows.append([sc.step, sc.n, f"{sc.mae:.3f}", f"{sc.rmse:.3f}", f"{sc.crps:.3f}",
                 f"{sc.coverage:.2f}", f"[{sc.coverage_region.lower}, {sc.coverage_region.upper}]",
                 str(sc.passed)])
table(rows, headers=("step", "n", "MAE", "RMSE", "CRPS", "coverage", "region", "passed"))
f0 = bt.forecasts[0]
assert isinstance(f0, OriginForecast)
print("origin", f0.origin, "periods", f0.periods, "observed[0]", np.round(f0.observed[0], 3))
print("failures:", [OriginFailure.model_validate(f.model_dump()) for f in bt.failures], "| passed:", bt.passed)
print("round-trips:", Backtest.from_json(bt.to_json()) == bt)

In [ ]:
backtest_plot(bt)

In [ ]:
origin_view = bt.forecasts[0]
unit = 0
steps = np.arange(len(origin_view.periods))
fig = curve_band(
    steps, origin_view.mean[unit], origin_view.lower[unit], origin_view.upper[unit],
    label="forecast",
    title="One origin, one unit, forecast against what happened",
    subtitle=f"refit on periods before {origin_view.origin}, then rolled forward through the same forward()",
    x_title="steps past the origin", y_title="outcome",
)
fig.add_scatter(x=steps, y=origin_view.observed[unit], mode="markers",
                marker={"size": 10, "color": ORANGE}, name="observed", showlegend=True)
caption(fig, "The carryover state entering step 0 is the one the training window actually "
             "ended in, not the one the test period implies. That distinction is what "
             "rolling_origin exists for — evaluating a model with memory on the horizon alone "
             "hands it information it would not have had.")

## Freeze and replay

`freeze` captures a fit's spec, posterior, and nuisance conventions in a `FrozenPredictor`
with a content hash; `predict(panel)` replays `prepare` + `forward` on any panel with the same
units and returns `PredictiveDraws` (or `Unsupported` for a unit mismatch).

In [ ]:
fitted = fit(carry.spec, carry.panel, draws=30, chains=1, seed=4)
fp = freeze(fitted)
assert isinstance(fp, FrozenPredictor)
print("hash:", fp.content_hash[:16], "| draws:", fp.provenance["n_draws"])
replay = fp.predict(carry.panel)
assert isinstance(replay, PredictiveDraws)
print("replayed shape:", replay.values.shape)
other = surface_world(n_units=4, n_periods=4, treatments=("a",), seed=1)
mismatch = fp.predict(other.panel)
assert isinstance(mismatch, Unsupported)
print("unit mismatch is typed:", mismatch.reason)

## What this bought you

Four refutation checks with stated pass rules and p-values that carry their N, and a backtest
that scores each horizon step with its coverage against an exact acceptance region — plus a
frozen predictor that can be replayed on a new panel, so "the model we shipped" is a hash
rather than a directory.